# 02 - Limpieza y preprocesamiento inicial del dataset AKI

## Objetivo
El objetivo de este notebook fue transformar el dataset explorado en el notebook anterior en un conjunto de datos limpio, coherente y preparado para fases posteriores de selección de variables y modelado predictivo. Para ello, se realizó una recarga reproducible del archivo original, la conversión correcta de tipos de datos, la exclusión de variables no aptas para modelado, la recodificación de variables categóricas y la creación de una versión procesada del dataset.

### 1. Importar librerías

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

FIGURES_DIR = "../figures"
RESULTS_DIR = "../results"

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Carpetas de figures y results listas.")

Carpetas de figures y results listas.


### 2. Deficinición de nobmbres de columnas y carga del dataset

In [2]:
columnas = [
    "IdIngreso",
    "UUIdPaciente",
    "Sexo",
    "EdadAproxIngreso",
    "FechaIngreso",
    "FechaAlta",
    "IdHospital",
    "IdSociedad",
    "IdDoctor",
    "PrimeraFechaLab",
    "Creatinina_mg_dL",
    "Urea_mg_dL",
    "Potasio_mmol_L",
    "Sodio_mmol_L",
    "BaselineFecha",
    "BaselineCreatinina",
    "MaxCreatinina48h",
    "AKI_48h",
    "AINE",
    "IECA",
    "ARAII",
    "DIURETICO",
    "VANCOMICINA",
    "AMINOGLUCOSIDO"
]

file_path = "C:/Users/isabe/OneDrive/Desktop/Resultados_SQL_TFG.csv"

df = pd.read_csv(
    file_path,
    sep=";",
    header=None,
    names=columnas,
    index_col=False,
    dtype=str
)

print("Dataset cargado correctamente.")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Dataset cargado correctamente.
Filas: 1175
Columnas: 24


### 3. Conversión correcta de tipos de datos

In [3]:
# Columnas de fecha
date_cols = ["FechaIngreso", "FechaAlta", "PrimeraFechaLab", "BaselineFecha"]

# Columnas numéricas decimales
float_cols = [
    "Creatinina_mg_dL",
    "Urea_mg_dL",
    "Potasio_mmol_L",
    "Sodio_mmol_L",
    "BaselineCreatinina",
    "MaxCreatinina48h"
]

# Columnas enteras y binarias
int_cols = [
    "IdIngreso",
    "Sexo",
    "EdadAproxIngreso",
    "IdHospital",
    "IdSociedad",
    "IdDoctor",
    "AKI_48h",
    "AINE",
    "IECA",
    "ARAII",
    "DIURETICO",
    "VANCOMICINA",
    "AMINOGLUCOSIDO"
]

# Conversión de fechas
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Conversión de variables decimales con coma decimal
for col in float_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
        .replace("nan", pd.NA)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Conversión de enteras y binarias
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

### 4. Verificación rápida de estructura, tipos y nulos

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1175 entries, 0 to 1174
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   IdIngreso           1175 non-null   Int64         
 1   UUIdPaciente        1175 non-null   str           
 2   Sexo                1175 non-null   Int64         
 3   EdadAproxIngreso    1175 non-null   Int64         
 4   FechaIngreso        1175 non-null   datetime64[us]
 5   FechaAlta           1150 non-null   datetime64[us]
 6   IdHospital          1175 non-null   Int64         
 7   IdSociedad          1175 non-null   Int64         
 8   IdDoctor            1175 non-null   Int64         
 9   PrimeraFechaLab     1175 non-null   datetime64[us]
 10  Creatinina_mg_dL    1170 non-null   float64       
 11  Urea_mg_dL          41 non-null     float64       
 12  Potasio_mmol_L      1122 non-null   float64       
 13  Sodio_mmol_L        1126 non-null   float64       
 14  Bas

In [5]:
missing_summary = pd.DataFrame({
    "n_missing": df.isnull().sum(),
    "percent_missing": (df.isnull().mean() * 100).round(2)
}).sort_values(by="percent_missing", ascending=False)

missing_summary

,n_missing,percent_missing
Urea_mg_dL,1134,96.51
MaxCreatinina48h,497,42.30
Potasio_mmol_L,53,4.51
Sodio_mmol_L,49,4.17
FechaAlta,25,2.13
Creatinina_mg_dL,5,0.43
BaselineFecha,2,0.17
BaselineCreatinina,2,0.17
IdSociedad,0,0.00
IdHospital,0,0.00


### 5. Definición de columnas no aptas para modelado

In [5]:
# Identificadores
id_cols = ["IdIngreso", "UUIdPaciente", "IdHospital", "IdSociedad", "IdDoctor"]

# Fechas
date_cols = ["FechaIngreso", "FechaAlta", "PrimeraFechaLab", "BaselineFecha"]

# Variable con fuga de información
leakage_cols = ["MaxCreatinina48h"]

# Variable problemática por missing extremo
high_missing_cols = ["Urea_mg_dL"]

# Target
target_col = "AKI_48h"

### 6. Justificación preliminar de exclusión de variables
En este paso se identificaron explícitamente los grupos de variables que no debían pasar al conjunto final de modelado:

#### Identificadores
- `IdIngreso`
- `UUIdPaciente`
- `IdHospital`
- `IdSociedad`
- `IdDoctor`

Estas variables fueron consideradas no aptas para modelado por no aportar significado clínico directo y por poder introducir ruido o patrones espurios.

#### Variables temporales
- `FechaIngreso`
- `FechaAlta`
- `PrimeraFechaLab`
- `BaselineFecha`

En esta fase no se utilizaron como predictores directos, ya que el enfoque del TFG prioriza un conjunto reducido de variables clínicas fácilmente interpretables.

#### Variable con riesgo de fuga de información
- `MaxCreatinina48h`

Esta variable fue excluida por formar parte de la ventana futura usada para definir el desenlace `AKI_48h`. Su inclusión como predictor habría supuesto una fuga de información metodológicamente inaceptable.

#### Variable con missing extremo
- `Urea_mg_dL`

Fue señalada como variable problemática debido a su porcentaje muy elevado de valores faltantes, lo que comprometía su utilidad analítica.

### 7. Revisión de la variable `Sexo` antes de recodificar
Antes de modificar la codificación de `Sexo`, se revisó su distribución de valores. Se confirmó que la variable estaba codificada en dos categorías:

- `1`
- `2`

y que no presentaba nulos ni categorías inesperadas.

Esta comprobación previa era necesaria para garantizar que la recodificación posterior se realizara sobre un esquema limpio y consistente.


In [6]:
print(df["Sexo"].value_counts(dropna=False))

Sexo
2    648
1    527
Name: count, dtype: Int64


### 8. Recodificaión de la variable `Sexo`
La variable `Sexo` fue transformada desde su codificación original `1/2` a una codificación binaria `0/1`, utilizando un mapeo técnico:
- `1 -> 0`
- `2 -> 1`

El objetivo de esta transformación fue facilitar su uso posterior en modelos de machine learning, sin alterar la información clínica subyacente. Tras la recodificación, se verificó de nuevo la distribución de la variable para confirmar que la transformación se había realizado correctamente.

In [7]:
# Recodificación técnica de Sexo: 1 -> 0, 2 -> 1
df["Sexo"] = df["Sexo"].map({1: 0, 2: 1}).astype("Int64")

print(df["Sexo"].value_counts(dropna=False))

Sexo
1    648
0    527
Name: count, dtype: Int64


### 9. Definición de variables candidatas para el dataset limpio
En este paso se definió el primer conjunto de variables candidatas para el modelado, compuesto por:

- `Sexo`
- `EdadAproxIngreso`
- `Creatinina_mg_dL`
- `Potasio_mmol_L`
- `Sodio_mmol_L`
- `BaselineCreatinina`
- `AINE`
- `IECA`
- `ARAII`
- `DIURETICO`
- `VANCOMICINA`
- `AMINOGLUCOSIDO`

A estas variables se añadió la variable objetivo:

- `AKI_48h`

Esta selección buscó equilibrar relevancia clínica, disponibilidad de datos y simplicidad metodológica, en línea con el enfoque prudente del proyecto.


In [8]:
candidate_features = [
    "Sexo",
    "EdadAproxIngreso",
    "Creatinina_mg_dL",
    "Potasio_mmol_L",
    "Sodio_mmol_L",
    "BaselineCreatinina",
    "AINE",
    "IECA",
    "ARAII",
    "DIURETICO",
    "VANCOMICINA",
    "AMINOGLUCOSIDO"
]

final_cols = candidate_features + [target_col]

### 10. Creación del dataset limpio
A partir de las variables seleccionadas, se construyó un nuevo DataFrame denominado `df_clean`, que contenía únicamente las columnas candidatas a modelado y la variable objetivo.

El dataset resultante presentó:
- **1175 filas**
- **13 columnas**

Esto confirmó que el número de observaciones se mantenía intacto y que el proceso de limpieza había reducido el conjunto de variables a una estructura compacta y clínicamente interpretable.

In [9]:
df_clean = df[final_cols].copy()

print("Dimensiones del dataset limpio:")
print(df_clean.shape)

df_clean.head()

Dimensiones del dataset limpio:
(1175, 13)


,Sexo,EdadAproxIngreso,Creatinina_mg_dL,Potasio_mmol_L,Sodio_mmol_L,BaselineCreatinina,AINE,IECA,ARAII,DIURETICO,VANCOMICINA,AMINOGLUCOSIDO,AKI_48h
0,0,61,0.96,4.83,144.0,0.96,0,0,0,0,0,0,0
1,0,46,1.15,4.41,142.0,1.15,0,0,0,0,0,0,0
2,1,25,0.75,4.09,138.0,0.75,1,0,0,0,0,0,0
3,0,78,0.97,4.70,143.0,0.97,0,0,0,0,0,0,0
4,1,35,0.63,4.43,140.0,0.63,0,0,0,0,0,0,0


### 11. Revisión de tipos y nulos en el dataset limpio
Se revisó la estructura de `df_clean` mediante `info()` y se generó un nuevo resumen de valores faltantes. Esta comprobación permitió confirmar que:

- las variables binarias y enteras estaban correctamente codificadas,
- las variables analíticas relevantes conservaban un número aceptable de observaciones,
- los valores faltantes residuales quedaban concentrados en pocas variables continuas.

Concretamente, se observó que:
- `Creatinina_mg_dL` presentaba 5 nulos,
- `Potasio_mmol_L` presentaba 53 nulos,
- `Sodio_mmol_L` presentaba 49 nulos,
- `BaselineCreatinina` presentaba 2 nulos.

Estas cifras se consideraron manejables y compatibles con estrategias posteriores de imputación o tratamiento específico.


In [10]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1175 entries, 0 to 1174
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Sexo                1175 non-null   Int64  
 1   EdadAproxIngreso    1175 non-null   Int64  
 2   Creatinina_mg_dL    1170 non-null   float64
 3   Potasio_mmol_L      1122 non-null   float64
 4   Sodio_mmol_L        1126 non-null   float64
 5   BaselineCreatinina  1173 non-null   float64
 6   AINE                1175 non-null   Int64  
 7   IECA                1175 non-null   Int64  
 8   ARAII               1175 non-null   Int64  
 9   DIURETICO           1175 non-null   Int64  
 10  VANCOMICINA         1175 non-null   Int64  
 11  AMINOGLUCOSIDO      1175 non-null   Int64  
 12  AKI_48h             1175 non-null   Int64  
dtypes: Int64(9), float64(4)
memory usage: 129.8 KB


In [11]:
clean_missing_summary = pd.DataFrame({
    "n_missing": df_clean.isnull().sum(),
    "percent_missing": (df_clean.isnull().mean() * 100).round(2)
}).sort_values(by="percent_missing", ascending=False)
clean_missing_summary.to_csv(f"{RESULTS_DIR}/results_02_clean_missing_summary.csv", index=True)
clean_missing_summary

,n_missing,percent_missing
Potasio_mmol_L,53,4.51
Sodio_mmol_L,49,4.17
Creatinina_mg_dL,5,0.43
BaselineCreatinina,2,0.17
Sexo,0,0.00
EdadAproxIngreso,0,0.00
AINE,0,0.00
IECA,0,0.00
ARAII,0,0.00
DIURETICO,0,0.00


### 12. Revisión de variables binarias y target en el dataset limpio
Se verificó nuevamente la distribución de:
- `Sexo`
- `AINE`
- `IECA`
- `ARAII`
- `DIURETICO`
- `VANCOMICINA`
- `AMINOGLUCOSIDO`
- `AKI_48h`

El propósito fue asegurar que, tras la limpieza y recodificación, todas estas variables mantenían una codificación correcta y consistente. Esta verificación confirmó que no se habían introducido errores durante el proceso y que el target permanecía intacto.


In [12]:
binary_check_cols = [
    "Sexo",
    "AINE",
    "IECA",
    "ARAII",
    "DIURETICO",
    "VANCOMICINA",
    "AMINOGLUCOSIDO",
    "AKI_48h"
]

for col in binary_check_cols:
    print(f"\n--- {col} ---")
    print(df_clean[col].value_counts(dropna=False))


--- Sexo ---
Sexo
1    648
0    527
Name: count, dtype: Int64

--- AINE ---
AINE
0    600
1    575
Name: count, dtype: Int64

--- IECA ---
IECA
0    1074
1     101
Name: count, dtype: Int64

--- ARAII ---
ARAII
0    1096
1      79
Name: count, dtype: Int64

--- DIURETICO ---
DIURETICO
0    703
1    472
Name: count, dtype: Int64

--- VANCOMICINA ---
VANCOMICINA
0    1152
1      23
Name: count, dtype: Int64

--- AMINOGLUCOSIDO ---
AMINOGLUCOSIDO
0    1166
1       9
Name: count, dtype: Int64

--- AKI_48h ---
AKI_48h
0    1146
1      29
Name: count, dtype: Int64


### 13. Revisión rápida de estadísticas descriptivas
Se aplicó `describe()` sobre `df_clean` para obtener una visión general de la distribución de las variables retenidas.

Este paso permitió comprobar que:
- los rangos de edad eran plausibles,
- las variables analíticas mantenían valores clínicamente razonables,
- las variables binarias se encontraban dentro de los rangos esperados,
- el dataset limpio era coherente y utilizable desde una perspectiva estadística básica.

In [13]:
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
Sexo,1175.0,0.551489,0.497554,0.0,0.0,1.0,1.0,1.0
EdadAproxIngreso,1175.0,71.82383,19.510285,1.0,63.0,76.0,86.0,110.0
Creatinina_mg_dL,1170.0,1.072846,0.719258,0.0,0.7,0.885,1.18,9.55
Potasio_mmol_L,1122.0,4.289929,0.624424,0.0,3.9,4.2,4.6,7.9
Sodio_mmol_L,1126.0,138.693872,6.664357,0.0,137.0,139.0,141.0,169.0
BaselineCreatinina,1173.0,1.072805,0.718568,0.0,0.7,0.88,1.18,9.55
AINE,1175.0,0.489362,0.5001,0.0,0.0,0.0,1.0,1.0
IECA,1175.0,0.085957,0.280421,0.0,0.0,0.0,0.0,1.0
ARAII,1175.0,0.067234,0.250534,0.0,0.0,0.0,0.0,1.0
DIURETICO,1175.0,0.401702,0.490451,0.0,0.0,0.0,1.0,1.0


### 14. Guardado del dataset limpio
Se creó la carpeta `data/processed/` y se guardó el dataset limpio con el nombre:

`aki_dataset_clean.csv`

Esta operación tuvo una importancia clave en términos de reproducibilidad del flujo de trabajo, ya que permite que los siguientes notebooks trabajen sobre una versión estable, documentada y previamente validada del conjunto de datos.


In [14]:
df_clean.describe(include="all").T.to_csv(f"{RESULTS_DIR}/results_02_clean_descriptive_summary.csv")

In [15]:
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "aki_dataset_clean.csv")

df_clean.to_csv(output_path, index=False)

print(f"Dataset limpio guardado en: {output_path}")

Dataset limpio guardado en: ../data/processed\aki_dataset_clean.csv


### 15. Conclusiones del notebook 02_data_cleaning
#### Conclusiones del proceso de limpieza inicial

En este notebook se realizó la limpieza y preparación inicial del dataset para su uso posterior en fases de selección de variables y modelado predictivo.

Las principales acciones realizadas fueron:

- recarga reproducible del dataset original desde el archivo CSV,
- conversión correcta de variables temporales, numéricas y binarias,
- exclusión de identificadores administrativos y clínicos no aptos para modelado,
- exclusión de variables con riesgo de fuga de información,
- exclusión provisional de `Urea_mg_dL` debido a su elevada proporción de valores faltantes,
- recodificación de la variable `Sexo` a formato binario,
- construcción de un dataset limpio con variables candidatas para modelado,
- guardado del conjunto procesado en la carpeta `data/processed`.

Como resultado, se obtuvo un dataset limpio y coherente, preparado para la siguiente fase del flujo de trabajo, centrada en la evaluación de variables y la construcción del dataset final de modelado.

- **1175 observaciones**
- **12 predictores clínicamente relevantes**
- **1 variable objetivo**
